# 01 — Run the strategy x density x seed experiment grid

Run `00_colab_setup.ipynb` first in this same Colab session (the CARLA server it starts must still be running). This notebook drives `experiment_grid.run_grid`, which is resumable: if Colab disconnects mid-run, just re-run this notebook — already-completed `(strategy, density, seed)` rows in `results/raw_episodes.csv` are skipped automatically.

In [ ]:
import sys
sys.path.insert(0, "/content/fag-project/src")

from merge_sim import carla_utils
from merge_sim.experiment_grid import run_grid, default_strategy_factories, RESULTS_PATH

client = carla_utils.connect(timeout=30.0)
print("Connected:", client.get_server_version())
print("Results will be written to:", RESULTS_PATH)

## Step 1 — smoke test with 1 seed per cell

Confirms the full pipeline (spawn -> negotiate/decide -> tick -> metrics -> CSV row) works end to end before committing Colab GPU time to the full grid. Expect ~3 strategies x 3 densities = 9 short episodes.

In [ ]:
run_grid(client, seeds=range(1))

In [ ]:
import pandas as pd
pd.read_csv(RESULTS_PATH)

## Step 2 — full grid (20 seeds/cell = 180 episodes for the 3 core strategies)

Only run this once the smoke test above looks correct (no all-collision or all-failure rows, sane TTC/jerk magnitudes). If a Colab runtime disconnects partway through, just re-run this cell — completed rows are skipped.

In [ ]:
run_grid(client, seeds=range(20))

## Optional — include the learned (RL) strategy

Only after training a policy with `merge_sim.rl_env.MergeEnv` (see the training snippet below) and saving it, e.g. to `/content/fag-project/models/ppo_merge.zip`.

In [ ]:
# from merge_sim.rl_env import MergeEnv
# from stable_baselines3 import PPO
#
# env = MergeEnv(client)
# model = PPO("MlpPolicy", env, verbose=1)
# model.learn(total_timesteps=50_000)
# model.save("/content/fag-project/models/ppo_merge")
# env.close()

In [ ]:
# factories = default_strategy_factories(learned_model_path="/content/fag-project/models/ppo_merge.zip")
# run_grid(client, strategy_factories=factories, seeds=range(20))